# Quantum ESPRESSO installation guide on Python for 2xT4 GPU

##  1. Install Dependencies

In [47]:
!apt-get update
!apt-get install -y build-essential wget \
  libfftw3-dev libblas-dev liblapack-dev \
  libscalapack-mpi-dev libopenmpi-dev openmpi-bin
!apt-get install -y libxc-dev libxc9
!pip install phonopy
!pip install numpy scipy pyyaml matplotlib

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 6,555 B in 1s (4,747 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2

##  2. Download Quantum ESPRESSO and Nvfortran

Will take ~15-17 GiB of output space, ensure there is enough storage

In [ ]:
!wget https://github.com/QEF/q-e/archive/refs/tags/qe-7.5.tar.gz
!tar -xvf qe-7.5.tar.gz
%cd q-e-qe-7.5

Here, nvfortran is a requirement otherwise the code cannot work. Use cuda 12.3 or above.

In [8]:
%cd /kaggle/working
!wget https://developer.download.nvidia.com/hpc-sdk/24.3/nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz
!tar -xvf nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz

/kaggle/working
--2026-04-30 14:18:08--  https://developer.download.nvidia.com/hpc-sdk/24.3/nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.46.228.170, 23.46.228.167
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.46.228.170|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5769368135 (5.4G) [application/x-gzip]
Saving to: ‘nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz’

nvhpc_2024_243_Linu 100%[===================>]   5.37G  61.9MB/s    in 32s     

2026-04-30 14:18:40 (174 MB/s) - ‘nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz’ saved [5769368135/5769368135]

nvhpc_2024_243_Linux_x86_64_cuda_12.3/install
nvhpc_2024_243_Linux_x86_64_cuda_12.3/install_components/
nvhpc_2024_243_Linux_x86_64_cuda_12.3/install_components/NVHPCConfig.cmake
nvhpc_2024_243_Linux_x86_64_cuda_12.3/install_components/NVHPCConfigVersion.cmake
nvhpc_2024_243_Linux_x86_64_cuda_12.3/instal

## 3. Setting environment for running 

This part is extremely crucial, **do not** skip. Without this step, the whole process is not applicable.

In [9]:
import os

base = "/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/install_components/Linux_x86_64/24.3" # Change path to current path where the nvhpc folder is

os.environ["PATH"] = base + "/compilers/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = base + "/compilers/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
print("Environment set.")
!nvfortran --version
print(" ")
print("Check for error")

Environment set.

nvfortran 24.3-0 64-bit target on Linux -tp skylake-avx512 
NVIDIA Compilers and Tools
Copyright (c) 2024, NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

nvfortran 24.3-0 64-bit target on x86-64 Linux -tp skylake-avx512 
NVIDIA Compilers and Tools
Copyright (c) 2024, NVIDIA CORPORATION & AFFILIATES.  All rights reserved.
 
Check for error


## 4. Set config 

Does a bit of cleanup to remove an extra zip file that has previously been extracted.

In [10]:
!rm -rf /kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz
!rm -rf /kaggle/working/qe-7.5.tar.gz

Ensure to not tinker with the config unless **absolutely needed**. **Do not modify.**

In [11]:
%cd /kaggle/working/q-e-qe-7.5 
# Change path as per need
!export LIBXC_LIBS="-L/usr/lib/x86_64-linux-gnu -lxc -lxcf03"
!export LIBXC_CFLAGS="-I/usr/include"
!./configure \
  FC=nvfortran \
  F90=nvfortran \
  F77=nvfortran \
  MPIF90=nvfortran \
  CC=nvc \
  CXX=nvc++ \
  LD=nvfortran \
  --with-cuda=yes \
  --with-cuda-runtime=12.3 \
  --with-cuda-cc=75 \
  --with-scalapack=no

/kaggle/working/q-e-qe-7.5
directory KCW/PP : okk ok okkokCI : ok
all dependencies updated successfully
Running configure as ./install/configure  FC=nvfortran F90=nvfortran F77=nvfortran MPIF90=nvfortran CC=nvc CXX=nvc++ LD=nvfortran --with-cuda=yes --with-cuda-runtime=12.3 --with-cuda-cc=75 --with-scalapack=no
configure: WARNING: you should use --build, --host, --target
checking build system type... x86_64-pc-linux-gnu
checking ARCH... x86_64
checking setting AR... ... ar
checking setting ARFLAGS... ... ruv
checking whether the Fortran compiler works... yes
checking for Fortran compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether we are using the GNU Fortran compiler... no
checking whether nvfortran accepts -g... yes
checking for Fortran flag to compile .f90 files... none
checking for nvfortran... nvfortran
checking whether we are using the GNU Fortran c

If final output shows configure: success, continue. Otherwise, search for documentation or use AI to help troubleshoot.

In [12]:
from pathlib import Path
import re

path = Path("/kaggle/working/q-e-qe-7.5/make.inc")
text = path.read_text()

new_lines = []
for line in text.splitlines():
    if line.strip().startswith("DFLAGS"):
        line = re.sub(r"-D__DFTI\b", "", line)
        line = re.sub(r"-D__FFTW3\b", "", line)
        line = re.sub(r"-D__FFTW\b", "", line)
        line = line.rstrip() + " -D__FFTW"
    new_lines.append(line)

path.write_text("\n".join(new_lines) + "\n")

print("Updated DFLAGS:")
for line in new_lines:
    if line.strip().startswith("DFLAGS"):
        print(line)

Updated DFLAGS:
DFLAGS         =  -D__PGI -D__CUDA -D__FFTW


## 5. Compile and Run

The below code may take a few minutes to run. **Do not** interrupt execution. In case of errors saying certain parts are not there/broken, remove the comment on the cell below the given one. Then run the below cell and re-comment.

In [13]:
'''with open('make.inc', 'r') as f:
    content = f.read()

import re
content = re.sub(r'^DFLAGS\s*=(.*)$', 
    lambda m: 'DFLAGS         =' + (' -D__FFTW3' if '__FFTW3' not in m.group(1) else '') + m.group(1),
    content, flags=re.MULTILINE)

with open('make.inc', 'w') as f:
    f.write(content)

with open('make.inc', 'r') as f:
    content = f.read()

import re
content = re.sub(r'^LIBMBD_LIBS\s*=.*$', 'LIBMBD_LIBS =', content, flags=re.MULTILINE)
content = re.sub(r'-D__MBD', '', content)

with open('make.inc', 'w') as f:
    f.write(content)
    
!cd /kaggle/working/q-e-qe-7.2/bin
!find /kaggle/working/q-e-qe-7.2 -name "*.mod" -delete
!find /kaggle/working/q-e-qe-7.2 -name "*.o" -delete'''
!cd /kaggle/working/q-e-qe-7.5/bin
!make pw -j2 2>&1
!make ph -j2 2>&1
!make pp -j2 2>&1

/bin/bash: line 1: cd: /kaggle/working/q-e-qe-7.5/bin: No such file or directory
test -d bin || mkdir bin
( cd UtilXlib ; make TLDEPS= all || exit 1 )
cd install ; make -f extlibs_makefile libcuda
make[1]: Entering directory '/kaggle/working/q-e-qe-7.5/UtilXlib'
nvfortran -fast -Mcache_align -Mpreprocess -Mlarge_arrays -D__PGI -D__CUDA -D__FFTW  -cuda -gpu=cc75,cuda12.3 -I/kaggle/working/q-e-qe-7.5//external/devxlib/src -I/kaggle/working/q-e-qe-7.5//external/devxlib/include -acc -I/kaggle/working/q-e-qe-7.5//external/devxlib/src -I. -I/kaggle/working/q-e-qe-7.5//include -I/opt/intel/mkl/include -I. -c parallel_include.f90 -o parallel_include.o
make[1]: Entering directory '/kaggle/working/q-e-qe-7.5/install'
initializing external/devxlib submodule ...


<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_57/1678561209.py:5: SyntaxWarning: invalid escape sequence '\s'
  content = re.sub(r'^DFLAGS\s*=(.*)$',


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /kaggle/working/q-e-qe-7.5/external/devxlib/.git/
nvfortran -fast -Mcache_align -Mpreprocess -Mlarge_arrays -D__PGI -D__CUDA -D__FFTW  -cuda -gpu=cc75,cuda12.3 -I/kaggle/working/q-e-qe-7.5//external/devxlib/src -I/kaggle/working/q-e-qe-7.5//external/devxlib/include -acc -I/kaggle/working/q-e-qe-7.5//external/devxlib/src -I. -I/kaggle/working/q-e-qe-7.5//include -I/opt/intel/mkl/include -I. -c nvtx_wrapper.f90 -o nvtx_wrapper.o
remote: Enumerating objects: 117, done.
remote: 

Below code runs QE and the analysis.

In [16]:
%%bash
cat > run_qe.sh << 'EOF'
#!/bin/bash
export CUDA_VISIBLE_DEVICES=${OMPI_COMM_WORLD_LOCAL_RANK}
exec /kaggle/working/q-e-qe-7.5/bin/pw.x -in input.in > output.out
EOF
chmod +x run_qe.sh

mpirun --allow-run-as-root -np 2 \
  --bind-to none \
  ./run_qe.sh || true

FORTRAN STOP
FORTRAN STOP


In [17]:
!cat output.out


     Program PWSCF v.7.5 starts on 30Apr2026 at 14:34:19 

     This program is part of the open-source Quantum ESPRESSO suite
     for quantum simulation of materials; please cite
         "P. Giannozzi et al., J. Phys.:Condens. Matter 21 395502 (2009);
         "P. Giannozzi et al., J. Phys.:Condens. Matter 29 465901 (2017);
         "P. Giannozzi et al., J. Chem. Phys. 152 154105 (2020);
          URL http://www.quantum-espresso.org", 
     in publications or presentations arising from this work. More details at
     http://www.quantum-espresso.org/quote

     Serial version
     1136 MiB available memory on the printing compute node when the environment starts
 
     Reading input from input.in

     Current dimensions of program PWSCF are:
     Max number of different atomic species (ntypx) = 10
     Max number of k-points (npk) =  40000
     Max angular momentum in pseudopotentials (lmaxx) =  4
     file Ti.pbe-spn-kjpaw_psl.1.0.0.UPF: wavefunction(s)  3S 3P 3D renormalized
    

Check outputs for any errors(saves a lot of time compared to trying to use mpirun error codes)

In [22]:
%%bash
cat > run_qe.sh << 'EOF'
#!/bin/bash
export CUDA_VISIBLE_DEVICES=${OMPI_COMM_WORLD_LOCAL_RANK}
exec /kaggle/working/q-e-qe-7.5/bin/d3hess.x -in d3.in > d3.out
EOF
chmod +x run_qe.sh

mpirun --allow-run-as-root -np 2 \
  --bind-to none \
  ./run_qe.sh || true

FORTRAN STOP
FORTRAN STOP


In [20]:
!cat d3.out


     Program POST-PROC v.7.5 starts on 30Apr2026 at 14:48:45 

     This program is part of the open-source Quantum ESPRESSO suite
     for quantum simulation of materials; please cite
         "P. Giannozzi et al., J. Phys.:Condens. Matter 21 395502 (2009);
         "P. Giannozzi et al., J. Phys.:Condens. Matter 29 465901 (2017);
         "P. Giannozzi et al., J. Chem. Phys. 152 154105 (2020);
          URL http://www.quantum-espresso.org", 
     in publications or presentations arising from this work. More details at
     http://www.quantum-espresso.org/quote

     Serial version
     8867 MiB available memory on the printing compute node when the environment starts
 

     Reading xml data from directory:

     ./tmp/calc.save/
     file Ti.pbe-spn-kjpaw_psl.1.0.0.UPF: wavefunction(s)  3S 3P 3D renormalized
     file N.pbe-n-kjpaw_psl.1.0.0.UPF: wavefunction(s)  2S renormalized
     file O.pbe-n-kjpaw_psl.1.0.0.UPF: wavefunction(s)  2S 2P renormalized

     IMPORTANT: XC functional

Runs ph.x phonon calculations.

In [44]:
%%bash
# 1. Mandatory cleanup of corrupted phonon data
rm -rf ./tmp/_ph0

# 2. Re-create the script with proper multi-GPU mapping
cat > run_qe.sh << 'EOF'
#!/bin/bash
# Lift stack limits for large perturbation arrays
ulimit -s unlimited
export CUDA_VISIBLE_DEVICES=${OMPI_COMM_WORLD_LOCAL_RANK}

# Launch ph.x with band parallelization (-nb 2) inside the script
# This distributes the 80+ Kohn-Sham states across both T4 GPUs
exec /kaggle/working/q-e-qe-7.5/bin/ph.x -nb 2 -in ph.in > ph.out
EOF
chmod +x run_qe.sh

# 3. Launch with 2 MPI ranks (one for each T4 GPU)
mpirun --allow-run-as-root -np 2 --bind-to none ./run_qe.sh || true

    1
    1
--------------------------------------------------------------------------
Primary job  terminated normally, but 1 process returned
a non-zero exit code. Per user-direction, the job has been aborted.
--------------------------------------------------------------------------
--------------------------------------------------------------------------
mpirun detected that one or more processes exited with non-zero status, thus causing
the job to be terminated. The first process to do so was:

  Process name: [[17444,1],1]
  Exit code:    1
--------------------------------------------------------------------------


In [45]:
!cat ph.out


     Program PHONON v.7.5 starts on 30Apr2026 at 15: 2:50 

     This program is part of the open-source Quantum ESPRESSO suite
     for quantum simulation of materials; please cite
         "P. Giannozzi et al., J. Phys.:Condens. Matter 21 395502 (2009);
         "P. Giannozzi et al., J. Phys.:Condens. Matter 29 465901 (2017);
         "P. Giannozzi et al., J. Chem. Phys. 152 154105 (2020);
          URL http://www.quantum-espresso.org", 
     in publications or presentations arising from this work. More details at
     http://www.quantum-espresso.org/quote

     Serial version
     7174 MiB available memory on the printing compute node when the environment starts
 
     Reading input from ph.in
      Title line not specified: using 'default'.

 %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
     Error in routine  read_namelists (1):
      bad line in namelist &inputph: "  ldisp = .false." (error could be in the previous line)
 %%%%%%%%%%%%%%%%%%%%%%%%

## 6.(Optional) In case .in files not present and only .xyz file available 

Run this if you have .in files and need ld1.x to run to create a .upf file.

In [ ]:
'''!cd /kaggle/working/q-e-qe-7.5/bin
!make ld1 -j2'''
!cd /kaggle/input/datasets/pcsciprav/actinide-basis-set-rel/fully-relativistic-basis
!../q-e-qe-7.2/bin/ld1.x < /kaggle/input/datasets/pcsciprav/actinide-basis-set-rel/fully-relativistic-basis/pu.in > pu.out
# Change the .in and .out file names to match the input .in file name. Use the same .out name.

Change the link to any pseudopotenntial file as an alternative to ld1.

In [14]:
!wget https://pseudopotentials.quantum-espresso.org/upf_files/As.pbe-n-kjpaw_psl.1.0.0.UPF
!wget https://pseudopotentials.quantum-espresso.org/upf_files/Ti.pbe-n-kjpaw_psl.1.0.0.UPF

--2026-04-30 14:32:43--  https://pseudopotentials.quantum-espresso.org/upf_files/As.pbe-n-kjpaw_psl.1.0.0.UPF
Resolving pseudopotentials.quantum-espresso.org (pseudopotentials.quantum-espresso.org)... 51.77.118.191
Connecting to pseudopotentials.quantum-espresso.org (pseudopotentials.quantum-espresso.org)|51.77.118.191|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1113750 (1.1M)
Saving to: ‘As.pbe-n-kjpaw_psl.1.0.0.UPF’

As.pbe-n-kjpaw_psl. 100%[===================>]   1.06M  1.26MB/s    in 0.8s    

2026-04-30 14:32:44 (1.26 MB/s) - ‘As.pbe-n-kjpaw_psl.1.0.0.UPF’ saved [1113750/1113750]

--2026-04-30 14:32:44--  https://pseudopotentials.quantum-espresso.org/upf_files/Ti.pbe-n-kjpaw_psl.1.0.0.UPF
Resolving pseudopotentials.quantum-espresso.org (pseudopotentials.quantum-espresso.org)... 51.77.118.191
Connecting to pseudopotentials.quantum-espresso.org (pseudopotentials.quantum-espresso.org)|51.77.118.191|:443... connected.
HTTP request sent, awaiting response

Run this snippet first and ensure persistence is set to files or variables and files.

In [15]:
import os
import shutil
import requests

PSEUDO_DIR = "pseudo"
os.makedirs(PSEUDO_DIR, exist_ok=True)
BASE_URL = "https://pseudopotentials.quantum-espresso.org/upf_files/"

manual_pseudos = {
    'As': '/kaggle/working/q-e-qe-7.5/As.pbe-n-kjpaw_psl.1.0.0.UPF'
}

masses = {
    'U': 238.03,
    'Pu': 244.06,
    'Au': 196.966570,
    'Fr': 223,
    'Cs': 132.91,
    'N': 14.007,
    'Nd': 144.242,
    'Tc': 97.907,
    'As': 74.9216
}

hubbard_u = {
    #'Nd': 6.0
}

def find_pseudo(element):
    return [
        f"{element}.pbe-spn-kjpaw_psl.1.0.0.UPF",
        f"{element}.pbe-n-kjpaw_psl.1.0.0.UPF",
        f"{element}.pbe-kjpaw_psl.1.0.0.UPF",
    ]

def try_download(element):
    for name in find_pseudo(element):
        path = os.path.join(PSEUDO_DIR, name)

        if os.path.exists(path):
            print(f"Using existing pseudo: {path}")
            return name

        url = BASE_URL + name
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and len(r.content) > 0:
                with open(path, "wb") as f:
                    f.write(r.content)
                print(f"Downloaded {name}")
                return name
        except Exception as e:
            print(f"Could not download {name}: {e}")

    return None

def install_manual_pseudo(element):
    src = manual_pseudos[element]
    if not os.path.exists(src):
        raise FileNotFoundError(f"Manual pseudopotential for {element} not found: {src}")

    filename = os.path.basename(src)
    dst = os.path.join(PSEUDO_DIR, filename)

    if not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print(f"Copied manual pseudo for {element} to {dst}")
    else:
        print(f"Using existing copied pseudo for {element}: {dst}")

    return filename

def download_pseudo(element):
    if element in manual_pseudos:
        return install_manual_pseudo(element)

    result = try_download(element)
    if result:
        return result

    raise Exception(f"No pseudopotential found for {element}")

def xyz_to_qe(xyz_file, output="input.in"):
    with open(xyz_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    nat = int(lines[0].strip())
    coords = lines[2:2 + nat]

    atoms = []
    species_ordered = []

    for line in coords:
        parts = line.split()
        if len(parts) < 4:
            raise ValueError(f"Invalid XYZ line: {line!r}")
        el, x, y, z = parts[:4]
        x = float(x) + 10.0
        y = float(y) + 10.0
        z = float(z) + 10.0
        atoms.append((el, x, y, z))
        if el not in species_ordered:
            species_ordered.append(el)

    pseudo_map = {}
    for el in species_ordered:
        pseudo_map[el] = download_pseudo(el)

    with open(output, "w", encoding="ascii", newline="\n") as f:
        # CONTROL
        f.write("&CONTROL\n")
        f.write("  calculation = 'scf'\n") # Either keep it as relax or scf
        f.write("  pseudo_dir = './pseudo/'\n")
        f.write("  outdir = './tmp/'\n")
        f.write("  prefix = 'calc'\n")
        f.write("/\n\n")

        # SYSTEM
        f.write("&SYSTEM\n")
        f.write(f"  nat = {nat}\n")
        f.write(f"  ntyp = {len(species_ordered)}\n")
        f.write("  ecutwfc = 65\n")
        #f.write("  ecutrho = 360\n")
        #f.write("  nbnd = 65\n")
        f.write("  input_dft = 'PBE'\n")
        f.write("  occupations = 'smearing'\n")
        f.write("  smearing = 'gaussian'\n")
        f.write("  degauss = 0.02\n")
        f.write("  vdw_corr = 'grimme-d3'\n")
        f.write("  dftd3_version = 4\n")
        f.write("  dftd3_threebody = .false.\n")
        f.write("  ibrav = 0\n")
        f.write("  nosym = .true.\n")
        f.write("  noinv = .true.\n")
        f.write("  noncolin = .false.\n")
        f.write("  lspinorb = .false.\n")
        f.write("/\n\n")

        # ELECTRONS
        f.write("&ELECTRONS\n")
        f.write("  conv_thr = 1.0d-8\n")
        f.write("  mixing_beta = 0.1\n")
        f.write("  electron_maxstep = 200\n")
        f.write("/\n\n")
        
        # IONS
        # Use only when running calculation = relax, otherwise comment it out.
        '''f.write("&IONS\n")
        f.write("  ion_dynamics = 'damp'\n")
        f.write("  trust_radius_ini = 0.10\n") # Lower for unstable systems
        f.write("  trust_radius_max = 0.20\n") # Lower for unstable systems
        f.write("/\n\n")'''

        # ATOMIC_SPECIES
        f.write("ATOMIC_SPECIES\n")
        for el in species_ordered:
            mass = masses.get(el, 1.0)
            pseudo_filename = pseudo_map[el]
            f.write(f"{el:>3}  {mass:10.5f}  {pseudo_filename}\n")
        
        # ATOMIC_POSITIONS
        f.write("\nATOMIC_POSITIONS angstrom\n")
        for el, x, y, z in atoms:
            f.write(f"{el:>3}  {x}  {y}  {z}\n")
        
        # CELL PARAMETERS ANGSTROMS
        f.write("\nCELL_PARAMETERS angstrom\n")
        f.write("18.0 0.0 0.0\n")
        f.write("0.0 18.0 0.0\n")
        f.write("0.0 0.0 18.0\n")

        # K_POINTS
        f.write("\nK_POINTS automatic\n")
        f.write("1 1 1 0 0 0\n")

        # HUBBARD
        has_hubbard = any(el in hubbard_u for el in species_ordered)
        if has_hubbard:
            f.write("\nHUBBARD ortho-atomic\n")
            for el in species_ordered:
                if el in hubbard_u:
                    f.write(f"U {el}-5f {hubbard_u[el]}\n")

    print(f"QE input ready: {output}")
    print("Pseudopotentials used:")
    for el in species_ordered:
        print(f"  {el} -> {pseudo_map[el]}")

xyz_to_qe("/kaggle/input/datasets/pcsciprav/xtbopt-example/compound.xyz")

Downloaded Ti.pbe-spn-kjpaw_psl.1.0.0.UPF
Downloaded N.pbe-n-kjpaw_psl.1.0.0.UPF
Copied manual pseudo for As to pseudo/As.pbe-n-kjpaw_psl.1.0.0.UPF
Downloaded O.pbe-n-kjpaw_psl.1.0.0.UPF
QE input ready: input.in
Pseudopotentials used:
  Ti -> Ti.pbe-spn-kjpaw_psl.1.0.0.UPF
  N -> N.pbe-n-kjpaw_psl.1.0.0.UPF
  As -> As.pbe-n-kjpaw_psl.1.0.0.UPF
  O -> O.pbe-n-kjpaw_psl.1.0.0.UPF


Generates input for d3hess.in

In [21]:
def generate_d3_input(prefix='calc', outdir='./tmp/'):
    """
    Generates the d3hess.x input file required to create the .hess file 
    for DFT-D3 phonon calculations.
    """
    d3_content = f"""&INPUT
    prefix = '{prefix}',
    outdir = '{outdir}',
/
"""
    with open('d3.in', 'w') as f:
        f.write(d3_content)
    print(f"Successfully created d3.in for prefix: {prefix}")

# Execution
generate_d3_input(prefix='calc', outdir='./tmp/')

Successfully created d3.in for prefix: calc


The below cell gives an input that can be fed to ph.x

In [43]:
def generate_ph_input_fixed(prefix="calc", outdir="./tmp/", verbosity="high"):
    with open("ph.in", "w") as f:
        f.write("&inputph\n")
        f.write(f"  prefix = '{prefix}',\n")
        f.write(f"  outdir = '{outdir}',\n")
        f.write("  tr2_ph = 1.0d-10,\n")
        f.write("  alpha_mix(1) = 0.10,\n")
        f.write("  nmix_ph = 8,\n")
        f.write("  diagonalization = 'cg',\n")
        f.write("  recover = .true.,\n")
        f.write("  niter_ph = 50,\n")
        f.write("  search_sym = .false.,\n")
        f.write("  fildyn = 'calc.dyn',\n")
        f.write("  trans = .true.,\n")
        #f.write("  ldisp = .false.,\n")
        f.write(f"  verbosity = '{verbosity}'\n")
        f.write("/\n")
        f.write("0.0 0.0 0.0\n")

    print("Generated ph.in for Gamma point molecular vibrational frequencies")

# Test single q-point first
generate_ph_input_molecule(single_q=True, verbosity='high')

Generated ph.in
mode: Gamma molecule
prefix: calc
outdir: ./tmp/


In [18]:
def write_pp_input_charge_density(filename="pp_charge.in"):
    with open(filename, "w") as f:

        # INPUTPP section
        f.write("&INPUTPP\n")
        f.write("   prefix = 'calc'\n")
        f.write("   outdir = './tmp/'\n")
        f.write("   filplot = 'charge_density'\n")
        f.write("   plot_num = 0\n")
        f.write("/\n\n")

        # PLOT section
        f.write("&PLOT\n")
        f.write("   iflag = 3\n")
        f.write("   output_format = 5\n")
        f.write("   fileout = 'charge_density.cube'\n")
        f.write("/\n")
write_pp_input_charge_density()

Used to create phonopy instance. Phonopy is much lighter than phonon.

In [61]:
import os
import re
import glob
import shutil
import subprocess
import sys
import yaml
from queue import Queue, Empty
from threading import Thread, Event, Lock

############################################################
# USER SETTINGS
############################################################

INPUT_FILE = "input.in"

QE_ROOT = "/kaggle/working/q-e-qe-7.5"
PW_COMMAND = os.path.join(QE_ROOT, "bin", "pw.x")

QE_TMP_ROOT = os.path.join(QE_ROOT, "tmp")
PHONOPY_TMP = os.path.join(QE_TMP_ROOT, "phonopy_fd")

MPI_EXE = "mpirun"
MPI_PROCS_PER_JOB = 1

GPUS = [0, 1]

DIM = "1 1 1"

CLEAN_OLD = True

############################################################
# SANITY CHECKS
############################################################

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Cannot find {INPUT_FILE}")

if not os.path.exists(PW_COMMAND):
    raise FileNotFoundError(f"Cannot find pw.x at {PW_COMMAND}")

os.makedirs(QE_TMP_ROOT, exist_ok=True)
os.makedirs(PHONOPY_TMP, exist_ok=True)

print("Using input file:", INPUT_FILE)
print("Using pw.x:", PW_COMMAND)
print("Using QE tmp root:", QE_TMP_ROOT)
print("Using Phonopy displacement tmp:", PHONOPY_TMP)
print("GPUs:", GPUS)

############################################################
# INSTALL PHONOPY AND PYYAML IF NEEDED
############################################################

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name

    try:
        __import__(import_name)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", pip_name],
            check=True
        )

ensure_package("phonopy")
ensure_package("yaml", "pyyaml")

############################################################
# CLEAN OLD GENERATED FILES
############################################################

if CLEAN_OLD:
    patterns = [
        "supercell*.in",
        "disp-*.in",
        "disp-*.out",
        "disp-*.err",
        "FORCE_SETS",
        "phonopy_disp.yaml",
        "phonopy.yaml",
        "qpoints.yaml",
        "gamma_frequencies_cm-1.txt",
    ]

    for pattern in patterns:
        for path in glob.glob(pattern):
            if os.path.isfile(path):
                os.remove(path)

    if os.path.isdir(PHONOPY_TMP):
        shutil.rmtree(PHONOPY_TMP)

    os.makedirs(PHONOPY_TMP, exist_ok=True)

############################################################
# ROBUST QE INPUT PARSER
############################################################

QE_CARDS = {
    "ATOMIC_SPECIES",
    "ATOMIC_POSITIONS",
    "CELL_PARAMETERS",
    "K_POINTS",
    "OCCUPATIONS",
    "CONSTRAINTS",
    "ATOMIC_FORCES",
    "HUBBARD",
}

def clean_line_for_parse(line):
    return line.split("!")[0].strip()

def first_token(line):
    clean = clean_line_for_parse(line)

    if not clean:
        return ""

    return clean.split()[0].upper()

def is_real_qe_card_start(line):
    clean = clean_line_for_parse(line)

    if not clean:
        return False

    if "=" in clean:
        return False

    token = first_token(clean)

    return token in QE_CARDS

def is_namelist_start(line):
    return clean_line_for_parse(line).startswith("&")

def get_header_only(qe_text):
    lines = qe_text.splitlines()
    out = []

    for line in lines:
        if is_real_qe_card_start(line):
            break

        out.append(line)

    return "\n".join(out).strip() + "\n"

def extract_cards(qe_text, wanted_cards):
    lines = qe_text.splitlines()
    out = []
    keep = False

    wanted_upper = set(x.upper() for x in wanted_cards)

    for line in lines:
        token = first_token(line)
        is_card = is_real_qe_card_start(line)

        if is_card and token in wanted_upper:
            keep = True
            out.append(line)
            continue

        if keep and (is_card or is_namelist_start(line)):
            if token not in wanted_upper:
                keep = False

        if keep:
            out.append(line)

    return "\n".join(out).strip() + "\n"

def patch_namelist(text, namelist, replacements):
    lines = text.splitlines()
    lower_name = "&" + namelist.lower()

    start = None
    end = None

    for i, line in enumerate(lines):
        if clean_line_for_parse(line).lower().startswith(lower_name):
            start = i
            break

    if start is None:
        block = [f"&{namelist}"]

        for key, value in replacements.items():
            block.append(f"  {key} = {value},")

        block.append("/")

        return "\n".join(block) + "\n" + text

    for j in range(start + 1, len(lines)):
        if clean_line_for_parse(lines[j]).startswith("/"):
            end = j
            break

    if end is None:
        print("\nProblematic namelist block:")
        for line in lines[start:start + 80]:
            print(line)

        raise RuntimeError(f"Namelist &{namelist} has no closing /")

    keys_to_replace = set(k.lower() for k in replacements)

    new_block = [lines[start]]

    for line in lines[start + 1:end]:
        stripped = clean_line_for_parse(line)

        if "=" in stripped:
            key = stripped.split("=")[0].strip().lower()
            key_base = key.split("(")[0].strip()

            if key in keys_to_replace or key_base in keys_to_replace:
                continue

        new_block.append(line)

    for key, value in replacements.items():
        new_block.append(f"  {key} = {value},")

    new_block.append(lines[end])

    return "\n".join(lines[:start] + new_block + lines[end + 1:]) + "\n"

def build_full_qe_input(supercell_file, job_index):
    with open(INPUT_FILE, "r") as f:
        original_text = f.read()

    with open(supercell_file, "r") as f:
        supercell_text = f.read()

    header = get_header_only(original_text)

    job_prefix = f"disp_{job_index:03d}"
    job_outdir = os.path.join(PHONOPY_TMP, job_prefix)
    os.makedirs(job_outdir, exist_ok=True)

    header = patch_namelist(
        header,
        "control",
        {
            "calculation": "'scf'",
            "restart_mode": "'from_scratch'",
            "prefix": f"'{job_prefix}'",
            "outdir": f"'{job_outdir}'",
            "tprnfor": ".true.",
            "tstress": ".false.",
        }
    )

    header = patch_namelist(
        header,
        "system",
        {
            "nosym": ".true.",
        }
    )

    structure = extract_cards(
        supercell_text,
        ["ATOMIC_SPECIES", "CELL_PARAMETERS", "ATOMIC_POSITIONS"]
    )

    if "ATOMIC_SPECIES" not in structure.upper():
        original_species = extract_cards(original_text, ["ATOMIC_SPECIES"])
        structure = original_species + "\n" + structure

    if "CELL_PARAMETERS" not in structure.upper():
        original_cell = extract_cards(original_text, ["CELL_PARAMETERS"])
        structure = original_cell + "\n" + structure

    if "ATOMIC_POSITIONS" not in structure.upper():
        raise RuntimeError(f"No ATOMIC_POSITIONS found in {supercell_file}")

    kpoints = "K_POINTS gamma\n"

    full_text = header.strip() + "\n\n" + structure.strip() + "\n\n" + kpoints

    output_input = f"disp-{job_index:03d}.in"

    with open(output_input, "w") as f:
        f.write(full_text)

    return output_input

def print_tail(path, n=80):
    if not os.path.exists(path):
        print(f"{path} does not exist")
        return

    with open(path, "r", errors="replace") as f:
        lines = f.readlines()

    print(f"\nLast {min(n, len(lines))} lines of {path}:\n")

    for line in lines[-n:]:
        print(line.rstrip())

############################################################
# GENERATE PHONOPY DISPLACEMENTS
############################################################

print("\nGenerating Phonopy displacements")

subprocess.run(
    [
        "phonopy",
        "--qe",
        "-d",
        f"--dim={DIM}",
        "-c",
        INPUT_FILE,
    ],
    check=True
)

supercell_files = sorted(glob.glob("supercell-*.in"))

if not supercell_files:
    raise RuntimeError("Phonopy generated no supercell-xxx.in files")

print("Generated displacement fragments:", len(supercell_files))

############################################################
# BUILD FULL QE INPUTS
############################################################

qe_inputs = []

for idx, sc_file in enumerate(supercell_files, start=1):
    full_input = build_full_qe_input(sc_file, idx)
    qe_inputs.append(full_input)

print("Built full QE displacement inputs:", len(qe_inputs))
print("Example full input:", qe_inputs[0])

############################################################
# RUN QE FORCE CALCULATIONS ON 2 GPUS
############################################################

job_queue = Queue()

for f in qe_inputs:
    job_queue.put(f)

failure_event = Event()
failures = []
print_lock = Lock()

def run_qe_job(infile, gpu_id):
    outfile = infile.replace(".in", ".out")
    errfile = infile.replace(".in", ".err")

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    cmd = [
        MPI_EXE,
        "--allow-run-as-root",
        "-x",
        "CUDA_VISIBLE_DEVICES",
        "-np",
        str(MPI_PROCS_PER_JOB),
        PW_COMMAND,
        "-i",
        infile,
    ]

    with open(outfile, "w") as out, open(errfile, "w") as err:
        result = subprocess.run(
            cmd,
            stdout=out,
            stderr=err,
            env=env,
        )

    if result.returncode != 0:
        raise RuntimeError(
            f"{infile} failed with exit code {result.returncode}"
        )

    with open(outfile, "r", errors="replace") as f:
        text = f.read()

    if "JOB DONE" not in text:
        raise RuntimeError(f"{infile} finished but JOB DONE was not found")

    if "Forces acting on atoms" not in text and "force =" not in text:
        raise RuntimeError(
            f"{infile} finished but forces were not clearly found"
        )

def worker(gpu_id):
    while not failure_event.is_set():
        try:
            infile = job_queue.get_nowait()
        except Empty:
            return

        with print_lock:
            print(f"GPU {gpu_id} running {infile}")

        try:
            run_qe_job(infile, gpu_id)

        except Exception as exc:
            failure_event.set()
            failures.append((infile, str(exc)))

            with print_lock:
                print("\nFAILED:", infile)
                print(str(exc))
                print_tail(infile.replace(".in", ".out"), n=80)
                print_tail(infile.replace(".in", ".err"), n=80)

        finally:
            job_queue.task_done()

threads = []

for gpu in GPUS:
    t = Thread(target=worker, args=(gpu,))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

if failures:
    raise RuntimeError(
        "At least one QE displacement calculation failed. "
        "The failing output was printed above."
    )

print("\nAll QE displacement calculations finished successfully")

############################################################
# COLLECT FORCE SETS
############################################################

out_files = sorted(glob.glob("disp-*.out"))

if len(out_files) != len(qe_inputs):
    raise RuntimeError(
        f"Expected {len(qe_inputs)} outputs, found {len(out_files)}"
    )

print("\nCollecting forces with Phonopy")

subprocess.run(
    ["phonopy", "--qe", "-f"] + out_files,
    check=True
)

if not os.path.exists("FORCE_SETS"):
    raise RuntimeError("FORCE_SETS was not created")

print("FORCE_SETS created")

############################################################
# COMPUTE GAMMA FREQUENCIES
############################################################

print("\nComputing Gamma point vibrational frequencies")

subprocess.run(
    [
        "phonopy",
        "--qe",
        "-c",
        INPUT_FILE,
        f"--dim={DIM}",
        "--qpoints=0 0 0",
        "--eigvecs",
    ],
    check=True
)

if not os.path.exists("qpoints.yaml"):
    raise RuntimeError("qpoints.yaml was not created")

with open("qpoints.yaml", "r") as f:
    data = yaml.safe_load(f)

bands = data["phonon"][0]["band"]

freq_thz = [band["frequency"] for band in bands]
freq_cm = [x * 33.3564095 for x in freq_thz]

with open("gamma_frequencies_cm-1.txt", "w") as f:
    f.write("# mode   THz              cm^-1\n")

    for i, (thz, cm) in enumerate(zip(freq_thz, freq_cm), start=1):
        f.write(f"{i:4d}   {thz:16.8f}   {cm:16.8f}\n")

print("\nGamma vibrational frequencies")
print("Mode        THz              cm^-1")

for i, (thz, cm) in enumerate(zip(freq_thz, freq_cm), start=1):
    print(f"{i:4d}   {thz:14.6f}   {cm:14.3f}")

print("\nSaved:")
print("FORCE_SETS")
print("qpoints.yaml")
print("gamma_frequencies_cm-1.txt")

Using input file: input.in
Using pw.x: /kaggle/working/q-e-qe-7.5/bin/pw.x
Using QE tmp root: /kaggle/working/q-e-qe-7.5/tmp
Using Phonopy displacement tmp: /kaggle/working/q-e-qe-7.5/tmp/phonopy_fd
GPUs: [0, 1]

Generating Phonopy displacements
        _
  _ __ | |__   ___  _ __   ___   _ __  _   _
 | '_ \| '_ \ / _ \| '_ \ / _ \ | '_ \| | | |
 | |_) | | | | (_) | | | | (_) || |_) | |_| |
 | .__/|_| |_|\___/|_| |_|\___(_) .__/ \__, |
 |_|                            |_|    |___/
                                       3.5.1

-------------------------[time 2026-04-30 15:26:07]-------------------------
Compiled with OpenMP support (max 4 threads).
Python version 3.12.12
Spglib version 2.7.0

Calculator interface: qe
Crystal structure was read from "input.in".
Unit of length: au
Displacements creation mode
  Systematic displacements
Settings:
  Supercell: [1 1 1]
Spacegroup: P1 (1)
Number of symmetry operations in supercell: 1
Use -v option to watch primitive cell, unit cell, and supercell

RuntimeError: At least one QE displacement calculation failed. The failing output was printed above.